# Python: External Interfaces

## 1. Operating system

### 1.1. File system
The `pathlib` library makes it easy to work with files and directories.

In [1]:
from pathlib import Path

In [2]:
p = Path('../data')
p.exists()

True

In [3]:
p.is_dir()

True

In [4]:
p.absolute()

PosixPath('/Users/hungpq/Documents/data-science/01_python_programming/../data')

In [5]:
(p / 'iris.csv').is_file()

True

In [6]:
list(p.glob('*.csv'))[:5]

[PosixPath('../data/cpi.csv'),
 PosixPath('../data/mall_customers.csv'),
 PosixPath('../data/credit_scoring.csv'),
 PosixPath('../data/air_quality.csv'),
 PosixPath('../data/macroeconomic.csv')]

### 1.2. Operating system

In [7]:
import os
print(
    os.environ['USER'],
    os.name,
    os.cpu_count(),
    sep='\n'
)

hungpq
posix
8


### 1.3. Logging
Logging is similar to `print()`, but provides structured and configurable event reporting. It supports severity levels, timestamps, file output, and selective filtering. Python provides the built-in [`logging`] module for this purpose.

[`Logging`]: https://docs.python.org/3/library/logging.html

#### Logging levels
The `logging` module defines five common [logging levels], listed below in ascending order of severity. Each level is represented by an integer constant, such as `logging.INFO`; using the named constants improves readability.

- `DEBUG`: Provides detailed diagnostic information, primarily for development and troubleshooting.
- `INFO`: Records normal application progress or significant runtime events. For example, "LightGBM outperformed the other candidates and was selected as the best model."
- `WARNING`: Indicates an unexpected or potentially problematic condition that does not prevent the program from continuing. Examples include detected data drift or an input column containing only zeros.
- `ERROR`: Indicates that an operation failed or could not be completed, such as when a required input file is missing.
- `CRITICAL`: Indicates a severe failure that may prevent the application from continuing.

Each level has a corresponding function for recording events. By default, the root logger processes messages at the `WARNING` level or higher. This threshold can be changed through the logging configuration.

[logging levels]: https://docs.python.org/3/library/logging.html#levels

In [8]:
import logging
logging.basicConfig(level=logging.INFO)

In [9]:
logging.debug('This is a debug message')
logging.info('This is an info message')
logging.warning('This is a warning message')
logging.error('This is an error message')
logging.critical('This is a critical message')

INFO:root:This is an info message
ERROR:root:This is an error message
CRITICAL:root:This is a critical message


In [ ]:
a = 5
b = 0

try:
    c = a / b
except ZeroDivisionError:
    logging.error("Exception occurred", exc_info=True)

ERROR:root:Exception occurred
Traceback (most recent call last):
  File "/var/folders/lf/svbf0gps7sd6gn_2jfw2p1y80000gn/T/ipykernel_4162/2671484370.py", line 5, in <module>
    c = a / b
        ~~^~~
ZeroDivisionError: division by zero


#### Customization
The [`basicConfig()`] function configures the root logger. Common options include:
- Set `filename` to write logs to a file instead of standard error. Use `filemode='w'` to overwrite the file or `filemode='a'` to append to it.
- Set `format` to define the log-record format. With `style='{'`, the format string uses `str.format()`-style placeholders. In general, these are called [logging attributes].
- Set `level` to define the minimum severity handled by the root logger.

[`basicConfig()`]: https://docs.python.org/3/library/logging.html#logging.basicConfig
[logging attributes]: https://docs.python.org/3/library/logging.html#logrecord-attributes

In [11]:
import logging
logging.basicConfig(
    level=logging.INFO,
    filename='../export/chap_01/mylog.log',
    filemode='w',
    style='{',
    format='{asctime} - {levelname}:{name} - {message}',
    datefmt='%H:%M:%S'
)

In [12]:
logging.debug('This is a debug message')
logging.info('This is an info message')
logging.warning('This is a warning message')
logging.error('This is an error message')
logging.critical('This is a critical message')

INFO:root:This is an info message
ERROR:root:This is an error message
CRITICAL:root:This is a critical message


#### Loggers
The `logging` module supports multiple named loggers for different components or purposes. `getLogger(name)` returns the logger associated with `name`; repeated calls with the same name return the same logger instance. For simpler configuration and formatting, consider the third-party [`loguru`] library.

[`loguru`]: https://github.com/Delgan/loguru

In [13]:
import logging

In [14]:
formatter = logging.Formatter(style='{', fmt='{asctime} - {levelname}:{name} - {message}')

handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
handler.setFormatter(formatter)

logger = logging.getLogger('my_logger')
logger.addHandler(handler)

In [15]:
logger.info('This is an info message')

2025-12-31 22:09:50,774 - INFO:my_logger - This is an info message
INFO:my_logger:This is an info message


In [ ]:
import sys
from loguru import logger
logger.remove()
logger.add(sys.stderr, format='{time} - {level} - {message}', level='INFO')
logger.info('Hello Loguru')

### 1.4. Arguments parsing
Command-line arguments allow script inputs - such as dates, file paths, and thresholds - to be changed without modifying the source code. Python's [`argparse`] module parses these arguments.

[`argparse`]: https://docs.python.org/3/library/argparse.html

#### Arguments
`argparse` supports positional and optional arguments. Positional arguments are identified by their order, whereas optional arguments are specified using option strings such as `-b` or `--beta`.

In [0]:
%%writefile ../export/chap_01/demo_argparse.py
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('alpha', type=int, default=1, nargs='?', help='first term')
parser.add_argument('-b', '--beta', type=int, default=2)
parser.add_argument('-g', '--gamma', type=int, default=3)
parser.add_argument('delta', type=int, default=4, nargs='?')
parser.add_argument('--sigma', action='store_true')
namespace = parser.parse_args()

print(
    f'alpha={namespace.alpha}, '
    f'beta={namespace.beta}, '
    f'gamma={namespace.gamma}, '
    f'delta={namespace.delta}, '
    f'sigma={namespace.sigma}'
)

#### Command line

In [0]:
!python ../export/chap_01/demo_argparse.py

In [0]:
!python ../export/chap_01/demo_argparse.py -h

In [0]:
!python ../export/chap_01/demo_argparse.py 10

In [0]:
!python ../export/chap_01/demo_argparse.py 10 1000 -b 20 -g 30

In [0]:
!python ../export/chap_01/demo_argparse.py 10 1000 --beta 20 --gamma 30 --sigma

## 2. Databases

### 2.1. SQL Server
The `pyodbc` library is used to connect to SQL Server.

In [0]:
import pandas as pd
import pyodbc

#### Connecting
Use a trusted connection for a local SQL Server instance. Otherwise, provide a username and password.

In [0]:
# local database information
driver = 'SQL Server Native Client 11.0'
server = ''
database = ''

# connect
conn = pyodbc.connect(
    f'Driver={{{driver}}};'
    f'Server={server};'
    f'Database={database};'
    f'Trusted_Connection=yes;'
)
cursor = conn.cursor()

In [0]:
# database with username and password
driver = 'SQL Server'
server = ''
database = ''
username = ''
password = ''

# connect
conn = pyodbc.connect(
    f'Driver={{{driver}}};'
    f'Server={server};'
    f'Database={database};'
    f'UID={username};'
    f'PWD={password};'
)
cursor = conn.cursor()

#### Listing all tables

In [0]:
cursor.execute('''
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'dbo'
''')

[table[0] for table in cursor.fetchall()]

#### Running a query

In [0]:
query = '''
SELECT TOP 5 *
FROM PROJECT_BUDGET_ADJ
'''

pd.read_sql_query(query, conn)

#### Reading an entire table

In [0]:
table = ''

pd.read_sql_query(f'SELECT * FROM {table}', conn)

### 2.2. PostgreSQL
The `psycopg2` library is used to connect to PostgreSQL.

#### Connecting

In [0]:
import pandas as pd
import psycopg2

In [0]:
# database information
host = ''
port = ''
database = ''
username = ''
password = ''

# connect
conn = psycopg2.connect(host=host, port=port, dbname=database, user=username, password=password)
cursor = conn.cursor()

#### Listing all tables

In [0]:
cursor.execute('''
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
AND table_name != 'user'
''')

[table[0] for table in cursor.fetchall()]

#### Running a query

In [0]:
query = '''
SELECT *
FROM bow
LIMIT 5
'''

pd.read_sql_query(query, connect)

#### Reading an entire table

In [0]:
table = ''

pd.read_sql_query(f'SELECT * FROM {table}', connect)

## 3. APIs
This section demonstrates the usage of [Clash Royale API](https://developer.clashroyale.com/#/documentation).
- Go to the Clash Royale API site and register an account.
- Find the device's *public IP address* by searching for "my ip" or "what is my ip" on Google. It is sometimes called *external IP*, and should not be confused with *local/internal IP*. Another way is to run `curl ifconfig.me` on either a macOS terminal or a Windows terminal.
- Open account settings and create a new key, using the *public IP* found earlier. This is the method Clash Royale API uses for authorizing the requests. After finished, copy the *token* associated with the key.
- Use the `requests` library to connect to the API and get response data.

In [18]:
!curl ifconfig.me

2001:ee0:4161:5eaa:d484:9bcd:914a:3374

In [1]:
import numpy as np
import pandas as pd
import requests
import json
pd.options.display.max_rows = 200

In [ ]:
url = 'https://api.clashroyale.com/v1/players/%232082RVQQ'
token = 'xxxxxxxx'  # replace with your actual token

headers = {
    'Accept': 'application/json',
    'Authorization': f'Bearer {token}'
}

response = requests.get(url=url, headers=headers)
response = response.json()
df = pd.DataFrame.from_dict(response['cards'])
df.head()

,name,id,level,starLevel,evolutionLevel,maxLevel,maxEvolutionLevel,rarity,count,elixirCost,iconUrls
0,Bomber,26000013,15,2.0,1.0,16,1.0,common,822,2.0,{'medium': 'https://api-assets.clashroyale.com...
1,Rascals,26000053,15,2.0,NaN,16,NaN,common,306,5.0,{'medium': 'https://api-assets.clashroyale.com...
2,Royal Recruits,26000047,15,2.0,1.0,16,1.0,common,704,7.0,{'medium': 'https://api-assets.clashroyale.com...
3,Minion Horde,26000022,15,3.0,NaN,16,NaN,common,394,5.0,{'medium': 'https://api-assets.clashroyale.com...
4,Royal Hogs,26000059,13,2.0,NaN,14,1.0,rare,101,5.0,{'medium': 'https://api-assets.clashroyale.com...


:::{admonition} Pitfall: Sensitive info
:class: danger

Never hardcode passwords and API tokens in notebooks. Use environment variables or a secrets manager.

:::